# Landing to Bronze — CineData Analytics
Ingestão dos 5 CSVs brutos + cotação do dólar (API do Banco Central) na camada Bronze,
sem alteração estrutural/de conteúdo, com `ingestion_datetime` e escrita em modo Append.

In [ ]:
# Notebook fino: toda a regra de negocio vive em src/, aqui so orquestramos.
# O notebook roda dentro de um Databricks Git Folder (Files in Repos), entao o
# diretorio de trabalho ja e code/notebooks/ -- subimos um nivel para achar code/
# e importar src/ como um pacote Python normal.
import os
import sys

sys.path.append(os.path.abspath(".."))

from datetime import date, timedelta

from src.bronze.ingest import adicionar_ingestion_datetime, cotacao_dolar_para_dataframe
from src.common.api_client import buscar_cotacao_dolar

dbutils.widgets.text("caminho_volume", "/Volumes/workspace/default/inputs/Inputs")
dbutils.widgets.text("data_inicio", (date.today() - timedelta(days=7)).isoformat())
dbutils.widgets.text("data_fim", date.today().isoformat())

caminho_volume = dbutils.widgets.get("caminho_volume")
data_inicio = date.fromisoformat(dbutils.widgets.get("data_inicio"))
data_fim = date.fromisoformat(dbutils.widgets.get("data_fim"))

In [ ]:
# Camada Bronze: dados brutos, sem qualquer alteracao estrutural ou de conteudo.
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

**Regras da camada Bronze (por que assim):**
- **Dado fiel à origem:** nenhuma limpeza aqui; a limpeza é responsabilidade da Silver. Assim a Bronze serve de auditoria e permite reprocessar.
- **`ingestion_datetime`:** instante da ingestão, usado depois para escolher o registro mais recente na deduplicação.
- **Delta + `append`:** exigência do enunciado. Cada execução acrescenta dados, então a Bronze acumula histórico; a Silver absorve as repetições.
- **Sem `multiLine`:** os CSVs têm aspas quebradas (texto de outras bases dentro das colunas). Com `multiLine=True` o Spark engolia várias linhas numa só; uma linha física = um registro preserva todos os dados.

In [ ]:
# Mapeamento arquivo -> tabela Bronze, conforme a secao 1.2 do enunciado (docs/stream.pdf).
mapeamento_arquivos = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews",
}

for nome_arquivo, tabela_destino in mapeamento_arquivos.items():
    # Bronze fiel a origem: uma linha fisica = um registro. NAO usar multiLine=True: os CSVs
    # trazem aspas quebradas (texto vazado de outras bases) e multiLine faz o Spark engolir
    # varias linhas numa so, perdendo registros (medido: ~4 mil linhas no metrics).
    df_bruto = spark.read.csv(
        f"{caminho_volume}/{nome_arquivo}",
        header=True,
        inferSchema=True,
        escape='"',
    )
    df_com_ingestao = adicionar_ingestion_datetime(df_bruto)
    df_com_ingestao.write.format("delta").mode("append").saveAsTable(tabela_destino)
    print(f"{tabela_destino}: {df_com_ingestao.count()} linhas gravadas")

**API de cotação do dólar (Banco Central):** consulta os últimos 7 dias a partir da execução (widgets `data_inicio` e `data_fim`), pois a API não tem cotação em fins de semana e feriados; o preenchimento desses dias acontece na Silver (forward fill).

In [ ]:
# Ingestao de API (obrigatoria): cotacao do dolar via API PTAX do Banco Central,
# para a diretoria financeira acompanhar orcamento/receita tambem em BRL.
registros_cotacao = buscar_cotacao_dolar(data_inicio, data_fim)
df_cotacao = cotacao_dolar_para_dataframe(spark, registros_cotacao)
df_cotacao = adicionar_ingestion_datetime(df_cotacao)
df_cotacao.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")

display(df_cotacao)